In [31]:
import numpy as np
import json
import re
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

In [16]:
# 1. Загрузка данных
newsgroups = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
docs = newsgroups.data
categories = newsgroups.target
cat_names = newsgroups.target_names

$$
p(w_i|C_i) = p(w_i|t)p(t|C_i) = \sum_t (phi_it * theta_it)
$$
$$
L = \sum_d \sum_w n_dw \log p(w|d) = \sum_i 1 * \log p(w_i|C_i)
$$

In [18]:
# 2. Агрегация текстов по категориям (1 "супер-документ" на тему)
cat_texts = []
for c in range(len(cat_names)):
    cat_docs = [docs[i] for i in range(len(docs)) if categories[i] == c]
    # Базовая очистка от сигнатур и email
    clean = " ".join(cat_docs)
    clean = re.sub(r'--\s*.*$', '', clean, flags=re.MULTILINE)
    clean = re.sub(r'\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b', '', clean)
    cat_texts.append(clean)

In [19]:
# 3. Векторизация на уровне категорий
vectorizer = TfidfVectorizer(
    max_features=15000,
    stop_words='english',
    min_df=1,           # всего 20 документов, порог частоты не нужен
    max_df=0.90,        # исключаем общеупотребимые слова
    sublinear_tf=True,
    token_pattern=r'(?u)\b[a-z]{2,}\b'  # только буквы, длина ≥2
)
cat_tfidf = vectorizer.fit_transform(cat_texts)
feature_names = vectorizer.get_feature_names_out()

In [24]:
# 4. Расчёт специфичности (якорности)
tfidf_matrix = cat_tfidf.toarray()
epsilon = 1e-8
anchor_scores = np.zeros_like(tfidf_matrix)

for cat in range(len(cat_names)):
    target = tfidf_matrix[cat]
    other_max = np.max(np.delete(tfidf_matrix, cat, axis=0), axis=0)
    # Специфичность = TF-IDF в целевой / (макс TF-IDF в других + eps)
    anchor_scores[cat] = target / (other_max + epsilon)

In [25]:
# 5. Извлечение топ-N якорных слов с фильтром по абсолютной значимости
N_ANCHORS = 25
anchors_dict = {}

for cat in range(len(cat_names)):
    # Сортируем по убыванию отношения специфичности
    top_idx = np.argsort(anchor_scores[cat])[::-1]
    # Берём просто топ-N без жёсткого MIN_TFIDF
    anchors_dict[cat_names[cat]] = feature_names[top_idx[:N_ANCHORS]].tolist()

In [28]:
# 6. Сохранение результатов
with open("20ng_anchors_tfidf.json", "w", encoding="utf-8") as f:
    json.dump(anchors_dict, f, ensure_ascii=False, indent=2)

print("✅ Готово. Пример (rec.sport.baseball):", anchors_dict.get("rec.sport.baseball", [])[:5])

✅ Готово. Пример (rec.sport.baseball): ['pitcher', 'hitter', 'alomar', 'phillies', 'dodgers']


In [33]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# ================= 1. КОНФИГУРАЦИЯ =================
K = 20                # Число якорных слов (латентных тем)
MAX_FEATURES = 10000  # Размер словаря
MIN_DF = 10           # Минимальная документная частота (фильтр шума)
MAX_DF = 0.85         # Максимальная документная частота (фильтр общих слов)
SMOOTHING_ALPHA = 1e-4 # Аддитивное сглаживание матрицы Q
N_CO_CONTEXT = 5      # Сколько контекстных слов выводить для валидации

# ================= 2. ЗАГРУЗКА И ПРЕПРОЦЕССИНГ =================
newsgroups = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
docs = newsgroups.data

# Кастомные стоп-слова для артефактов newsgroup + стандартные английские
NG_ARTIFACTS = {
    "sig", "sub", "subject", "organization", "distribution", "nntp", 
    "posting", "host", "lines", "path", "expires", "reply", "thanks", 
    "appreciate", "write", "article", "please", "help"
}
CUSTOM_STOPS = ENGLISH_STOP_WORDS.union(NG_ARTIFACTS)

vectorizer = CountVectorizer(
    max_features=MAX_FEATURES,
    stop_words=list(CUSTOM_STOPS),
    min_df=MIN_DF,
    max_df=MAX_DF,
    token_pattern=r'(?u)\b[a-z]{3,}\b'  # Только буквы, длина ≥3
)
X = vectorizer.fit_transform(docs)
feature_names = vectorizer.get_feature_names_out()
V = len(feature_names)

print(f"📦 Словарь: {V} слов | Документов: {X.shape[0]}")

# ================= 3. МАТРИЦА УСЛОВНЫХ ВЕРОЯТНОСТЕЙ =================
# L1-нормализация строк → эмпирическое распределение P(w | d)
doc_probs = normalize(X.astype(float), norm='l1', axis=1, copy=True)

# Совместные вероятности: Q_ij ≈ Σ_d P(w_i|d) P(w_j|d)
Q = doc_probs.T.dot(doc_probs).toarray()  # shape: (V, V)

# Аддитивное сглаживание (стабилизирует геометрию симплекса)
Q += SMOOTHING_ALPHA

# Условные вероятности: P(w_j | w_i) = Q_ij / Σ_k Q_ik
row_sums = Q.sum(axis=1, keepdims=True)
P = Q / row_sums

# ================= 4. SPA (ПОИСК ВЕРШИН СИМПЛЕКСА) =================
anchors_idx = []
residual = P.copy()

for k in range(K):
    # Жадный выбор строки с максимальной L2-нормой в остатке
    norms = np.linalg.norm(residual, axis=1)
    
    # Защита от дубликатов из-за численных погрешностей
    for idx in anchors_idx:
        norms[idx] = -1.0
    anchor = np.argmax(norms)
    anchors_idx.append(anchor)
    
    # Ортогональная проекция: удаляем компоненту найденного якоря
    a_vec = residual[anchor]
    a_norm_sq = np.dot(a_vec, a_vec)
    if a_norm_sq < 1e-12:
        print(f"⚠️ Ранняя остановка на итерации {k} (норма близка к нулю).")
        break
        
    # P_proj = (R @ a) * a^T / ||a||^2
    proj = np.outer(residual @ a_vec, a_vec) / a_norm_sq
    residual -= proj

# ================= 5. ВЫВОД И ВАЛИДАЦИЯ =================
anchor_words = [feature_names[i] for i in anchors_idx]
print("\n🔑 Извлечённые якорные слова:")
for i, w in enumerate(anchor_words):
    print(f"  Тема {i:02d}: {w}")

print(f"\n📊 Валидация: топ-{N_CO_CONTEXT} слов, наиболее коррелирующих с каждым якорем")
for idx, word in zip(anchors_idx, anchor_words):
    # Исключаем само слово (индекс 0 после сортировки)
    top_co = np.argsort(P[idx])[::-1][1:N_CO_CONTEXT+1]
    context = ", ".join(feature_names[i] for i in top_co)
    print(f"  {word:15} → {context}")

# Сохранение результатов (опционально)
import json
output = {
    "anchors": anchor_words,
    "parameters": {"K": K, "MAX_FEATURES": MAX_FEATURES, "MIN_DF": MIN_DF, "MAX_DF": MAX_DF},
    "validation_context": {
        anchor_words[i]: [feature_names[j] for j in np.argsort(P[idx])[::-1][1:N_CO_CONTEXT+1]]
        for i, idx in enumerate(anchors_idx)
    }
}
with open("20ng_anchors_arora_spa.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
print("\n💾 Результаты сохранены в 20ng_anchors_arora_spa.json")

📦 Словарь: 9128 слов | Документов: 11314

🔑 Извлечённые якорные слова:
  Тема 00: ditto
  Тема 01: deletion
  Тема 02: grandfather
  Тема 03: max
  Тема 04: moments
  Тема 05: ironic
  Тема 06: chocolate
  Тема 07: test
  Тема 08: corn
  Тема 09: borders
  Тема 10: methodology
  Тема 11: maine
  Тема 12: oops
  Тема 13: spelling
  Тема 14: keywords
  Тема 15: ass
  Тема 16: notion
  Тема 17: spell
  Тема 18: raise
  Тема 19: update

📊 Валидация: топ-5 слов, наиболее коррелирующих с каждым якорем
  ditto           → does, time, just, used, years
  deletion        → god, alt, atheism, religion, say
  grandfather     → like, clock, tiger, beautiful, size
  max             → giz, bhj, bxn, qax, nrhj
  moments         → ftp, ago, dark, screw, just
  ironic          → people, don, period, knew, friends
  chocolate       → taste, like, bar, strawberry, space
  test            → message, used, want, just, really
  corn            → edu, having, seizure, gordon, cause
  borders         → israel